# Chapter 27: More on Models and Numerical Procedures

> **Color Convention**  
> 🟡 **Yellow** = Definition / Theorem / Property  
> 🟢 **Green** = Comment / Application / Insight

---

## Overview

This chapter extends BSM with alternative asset price models and develops more sophisticated numerical methods for exotic options. The key problem: BSM's volatility surface calibrates to vanilla options but may misprice exotic options.

---

## 27.1 Alternatives to Black–Scholes–Merton

🟢 **Comment:**

> BSM assumes geometric Brownian motion → lognormal distribution. Limitations:
> - Cannot simultaneously match all strikes in the volatility smile
> - Inappropriate for many exotic options
>
> Three categories of alternatives:
> 1. Continuous diffusions (other than GBM)
> 2. Jump-diffusion models
> 3. Pure jump models

🟢 **Comment:**

> Three models implemented in DerivaGem: CEV, Merton jump-diffusion, and variance-gamma. All three are **jump or diffusion-based** and produce non-lognormal distributions, better fitting market smiles.

### Constant Elasticity of Variance (CEV) Model

🟡 **Definition:**

> $$dS = (r-q)S\,dt + \sigma S^\beta\,dz$$
>
> - $\beta < 1$: volatility decreases as $S$ rises → **negative skew** (fits equities)
> - $\beta > 1$: volatility increases as $S$ rises → **positive skew**
> - $\beta = 1$: standard BSM (GBM)

### Merton's Jump-Diffusion Model

🟡 **Definition:**

> Asset follows GBM plus compound Poisson jumps:
>
> $$\frac{dS}{S} = (r - q - \lambda k)\,dt + \sigma\,dz + dp$$
>
> - $\lambda$ = average number of jumps per year
> - $k = E[J-1]$ = average proportional jump size
> - $dp$ = Poisson process component: jumps of size $J-1$

🟡 **Parameters (Merton):**

> $\lambda$: jump frequency, $k$: average jump size, $\sigma$: GBM volatility, plus parameters of the jump size distribution (lognormal jump amplitude).
>
> Option price = weighted sum of BSM prices conditional on exactly $n = 0, 1, 2, \ldots$ jumps.

### Variance-Gamma Model

🟡 **Definition:**

> A **pure jump** model. Let $g$ = change over time $T$ in a gamma process with mean rate 1 and variance rate $v$. The log stock price change:
>
> $$\ln(S_T/S_0) = \theta g + \sigma\sqrt{g}\,\epsilon, \quad \epsilon \sim N(0,1)$$
>
> This produces **asymmetric, leptokurtic distributions** with three parameters: $\sigma$ (volatility), $\nu$ (variance rate of gamma process), $\theta$ (skewness parameter).

---

## 27.2 Stochastic Volatility Models

🟢 **Comment:**

> When volatility is stochastic and uncorrelated with $S$, the European option price = BSM price integrated over the distribution of average variance:
>
> $$c = \int_0^\infty c_{BSM}(\bar{V}) p(\bar{V})\,d\bar{V}$$
>
> where $\bar{V}$ = mean variance over $[0,T]$.

🟢 **Comment:**

> Hull–White stochastic vol model: $dV = a(V_L - V)\,dt + \xi V\,dz_V$ where $V$ mean-reverts to $V_L$.

### SABR Model

🟡 **Definition:**

> Very popular for interest rate options:
>
> $$dF = \sigma F^\beta\,dz, \quad d\sigma = \nu\sigma\,dw$$
>
> $\text{corr}(dz, dw) = \rho$. Parameters: $\alpha$ (initial vol), $\beta$ (CEV exponent), $\rho$ (correlation), $\nu$ (vol of vol).
>
> Provides a **closed-form approximation** for implied volatility as a function of strike and maturity, making calibration easy.

🟡 **Property:**

> SABR is particularly useful for managing smile dynamics through time — capturing how the smile shifts as the forward rate changes.

### Rough Volatility Models

🟡 **Property:**

> Volatility modeled using **fractional Brownian motion** with Hurst exponent $H < 0.5$:
> - $H = 0.5$: standard BM
> - $H < 0.5$: rougher paths, better matching empirical volatility time series
>
> Well-known example: **Rough Heston model**, which outperforms standard stochastic vol models.

---

## 27.3 The IVF / Local Volatility Model

🟡 **Definition:**

> **Dupire's local volatility (IVF) model** (Derman–Kani–Rubinstein, 1994): A BSM-type model with **deterministic, state- and time-dependent volatility** $\sigma(S, t)$ that *exactly* fits the entire observed vanilla volatility surface.
>
> The local volatility function extracted from call prices:
>
> $$\sigma^2(K, T) = \frac{\partial c/\partial T + (r-q)K\,\partial c/\partial K + qc}{\frac{1}{2}K^2\,\partial^2 c/\partial K^2}$$

🟡 **Property:**

> The IVF model provides a unique, internally consistent model calibrated to all vanilla option prices. For exotic options, it prices consistently with the observed vanilla smile. However, it may predict incorrect dynamics for the smile itself (implied vol surface dynamics).

---

## 27.4 Convertible Bonds

🟡 **Definition:**

> A **convertible bond** allows the holder to exchange the bond for company stock at specified times using a **conversion ratio** (shares per bond). Value:
>
> $$V_{\text{node}} = \max\left(\min\left(\text{hold value}, \text{call price}\right), \text{conversion value}\right)$$
>
> Valuation: binomial tree incorporating both equity risk (for conversion) and credit risk (for default). The discount rate shifts between risk-free (when equity-like) and risky (when bond-like).

---

## 27.5 Path-Dependent Derivatives

🟡 **Definition:**

> A **path-dependent derivative** has a payoff depending on the history of $S$, not just the final value. Examples: Asian options, lookback options, barrier options.

🟢 **Application:**

> For path-dependent derivatives satisfying two conditions:
> 1. Payoff depends on a single path function $F$ (e.g., running average or maximum)
> 2. $F$ can be updated analytically at each tree step
>
> → Extend binomial tree by tracking $F$ at each node. Interpolate between tracked values.

---

## 27.6 Barrier Options (Numerical Issues)

🟡 **Property:**

> Binomial trees give **inaccurate results** for barrier options because the discrete barrier (formed by tree nodes) differs from the continuous true barrier. The error is $O(\sqrt{\Delta t})$ — very slow convergence.

🟢 **Application (Inner/Outer Barrier Interpolation):**

> 1. Compute price assuming the **inner barrier** (nodes just inside the true barrier) is the true barrier
> 2. Compute price assuming the **outer barrier** (nodes just outside) is the true barrier
> 3. Interpolate using distances to the true barrier
>
> This reduces the error dramatically.

---

## 27.7 Options on Two Correlated Assets

🟢 **Application:**

> For two **uncorrelated** variables: build a 3D tree by combining two 2D trees.
>
> For two **correlated** variables with correlation $\rho$: transform to uncorrelated coordinates:
>
> $$y_1 = x_1, \quad y_2 = \frac{x_2 - \rho x_1}{\sqrt{1-\rho^2}}$$
>
> where $x_i = F_i / \sigma_i$. Build the tree in $(y_1, y_2)$ space, then transform back.

---

## 27.8 Monte Carlo Simulation and American Options

### Parameterization Approach

🟢 **Application:**

> Parameterize the early exercise boundary and optimize parameters iteratively starting from the option's end date. Systematic but computationally intensive.

### Longstaff–Schwartz Least Squares Method

The most widely used Monte Carlo approach for American options:

1. Simulate $N$ paths forward under the risk-neutral measure
2. At each exercise date $t_i$ (working backward from $T$):
   - Identify in-the-money paths
   - Regress continuation value on basis functions of $S_{t_i}$ (e.g., $1, S, S^2$)
   - Compare regression estimate of continuation value to immediate exercise value
   - Exercise where immediate exercise > estimated continuation
3. Discount backward along each path

🟢 **Application:**

> The Longstaff–Schwartz method is the industry standard for pricing American-style securities in Monte Carlo. Works for multi-dimensional problems (basket Americans, Bermudans) where trees are infeasible.

---

## Summary

### Alternative Models

| Model | Key Feature |
|-------|-------------|
| CEV | $\sigma \propto S^{\beta-1}$; captures the smile skew |
| Jump-diffusion (Merton) | GBM + Poisson jumps; heavy tails and negative skew |
| Variance-gamma | Pure jump model; flexible skew and kurtosis |
| SABR | Stochastic vol; tractable smile parameterization for rates |
| Local vol (Dupire/IVF) | Exactly fits the full vanilla vol surface |
| Rough volatility | Fractional BM ($H < 0.5$); better matches empirical vol |

### Numerical Issues and Solutions

| Problem | Solution |
|---------|---------|
| Barrier options (tree inaccuracy) | Inner/outer barrier interpolation |
| Path-dependent derivatives | Track path function $F$ at tree nodes |
| Two correlated assets (tree) | Transform to uncorrelated coordinates |
| American options (Monte Carlo) | Longstaff–Schwartz least squares regression |
